<a href="https://colab.research.google.com/github/kuroshkarimi/Machine-Learning-and-Data-Analysis/blob/main/Deep_Learning/Predicting_income_category.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting income category

The task is to develop a binary classification model that predicts whether a person’s annual income exceeds 50,000 dollars, using the Adult dataset, downloaded through scikit-learn’s OpenML interface. This project focuses on preparing numerical and categorical features, training a neural network, and comparing its performance with logistic regression.
Each row represents one person. Input features include the followings:

| Column | Meaning |
|---|---|
| **status** | **marital-status**: married, divorced, never married, widowed, etc. |
| **occupation** | Job category, such as sales, technical support, or managerial work. |
| **relationship** | Household or family role, such as husband, wife, own child, or other relative. |
| **race** | Recorded racial category. |
| **sex** | Recorded sex; the original dataset uses Female and Male. |
| **capital-gain** | Reported monetary gains from selling capital assets, such as investments. |
| **capital-loss** | Reported monetary losses from selling capital assets. |
| **hours-per-week** | Number of hours worked per week. |
| **native-country** | Country of origin, generally interpreted as country of birth. |
| **age** | Person’s age in years. |
| **workclass** | Employment category, such as private-sector, government, or self-employed. |
| **fnlwgt** | “Final weight”: a survey weight indicating how many people in the population the record represents. It is not income. |
| **education** | Highest recorded education level, such as high-school graduate, bachelor’s, or master’s. |
| **education-num** | An ordered numerical code for education level. It should not automatically be interpreted as years spent in education. |

The target is:
- 0: Annual income is at or below 50,000 dollars.
- 1: Annual income is above $50,000.
For each person, the model should produce an estimated probability of belonging to class 1. A chosen threshold then converts that score into a predicted income category.
Your implementation should accomplish the following:
1. Explore the dataset: examine feature types, missing values, class proportions, and relationships between features and income category.
2. Split the data: allocate 60% for training, 20% for validation, and 20% for testing, preserving approximately the same class proportions.
3. Prepare the inputs: impute missing numerical values, standardize numerical features, and one-hot encode categorical features. Fit preprocessing exclusively on the training set.
4. Establish a baseline: train logistic regression using the prepared features.
5. Train neural networks: compare the three proposed configurations—one hidden layer with 32 neurons, two layers with 64 and 32 neurons, and the same two-layer model with dropout.
6. Select the model: compare validation average precision, validation loss, and training time. Select a decision threshold on validation data if a specific precision–recall trade-off is required.
7. Evaluate the final model: report test average precision, ROC AUC, precision, recall, F1, and a confusion matrix.
8. Demonstrate prediction: pass a raw observation through the fitted preprocessing steps and obtain its predicted probability and category.
The main research question is:
Does a dense neural network improve income classification enough to justify its additional complexity compared with logistic regression?

A successful project does not require the neural network to win. It requires a fair comparison, preprocessing without data leakage, and a clear explanation of the results.
The final deliverable should be a notebook containing the analysis, preprocessing, training, evaluation, and conclusions. Because the dataset is historical, interpret the results as performance on this dataset—not evidence that the model reliably predicts present-day incomes.

In [38]:
from sklearn.datasets import fetch_openml

In [39]:
adult = fetch_openml(
    name='adult',
    version=2,
    as_frame=True,

)
x = adult.data.copy()
target = adult.target.copy()

In [40]:
print(x.head(2))
print("\n")
print(target.head(2))

   age workclass  fnlwgt education  education-num      marital-status  \
0   25   Private  226802      11th              7       Never-married   
1   38   Private   89814   HS-grad              9  Married-civ-spouse   

          occupation relationship   race   sex  capital-gain  capital-loss  \
0  Machine-op-inspct    Own-child  Black  Male             0             0   
1    Farming-fishing      Husband  White  Male             0             0   

   hours-per-week native-country  
0              40  United-States  
1              50  United-States  


0    <=50K
1    <=50K
Name: class, dtype: category
Categories (2, object): ['<=50K', '>50K']


In [41]:
x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   age             48842 non-null  int64   
 1   workclass       46043 non-null  category
 2   fnlwgt          48842 non-null  int64   
 3   education       48842 non-null  category
 4   education-num   48842 non-null  int64   
 5   marital-status  48842 non-null  category
 6   occupation      46033 non-null  category
 7   relationship    48842 non-null  category
 8   race            48842 non-null  category
 9   sex             48842 non-null  category
 10  capital-gain    48842 non-null  int64   
 11  capital-loss    48842 non-null  int64   
 12  hours-per-week  48842 non-null  int64   
 13  native-country  47985 non-null  category
dtypes: category(8), int64(6)
memory usage: 2.6 MB


In [42]:
x.describe(include = 'all')

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
count,48842.000000,46043,4.884200e+04,48842,48842.000000,48842,46033,48842,48842,48842,48842.000000,48842.000000,48842.000000,47985
unique,NaN,8,NaN,16,NaN,7,14,6,5,2,NaN,NaN,NaN,41
top,NaN,Private,NaN,HS-grad,NaN,Married-civ-spouse,Prof-specialty,Husband,White,Male,NaN,NaN,NaN,United-States
freq,NaN,33906,NaN,15784,NaN,22379,6172,19716,41762,32650,NaN,NaN,NaN,43832
mean,38.643585,NaN,1.896641e+05,NaN,10.078089,NaN,NaN,NaN,NaN,NaN,1079.067626,87.502314,40.422382,NaN
std,13.710510,NaN,1.056040e+05,NaN,2.570973,NaN,NaN,NaN,NaN,NaN,7452.019058,403.004552,12.391444,NaN
min,17.000000,NaN,1.228500e+04,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,1.000000,NaN
25%,28.000000,NaN,1.175505e+05,NaN,9.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,40.000000,NaN
50%,37.000000,NaN,1.781445e+05,NaN,10.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,40.000000,NaN
75%,48.000000,NaN,2.376420e+05,NaN,12.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,45.000000,NaN


In [43]:
target.info()

<class 'pandas.core.series.Series'>
RangeIndex: 48842 entries, 0 to 48841
Series name: class
Non-Null Count  Dtype   
--------------  -----   
48842 non-null  category
dtypes: category(1)
memory usage: 47.9 KB


In [44]:
print(target.value_counts())
target.value_counts(normalize = True)

class
<=50K    37155
>50K     11687
Name: count, dtype: int64


,proportion
class,
<=50K,0.760718
>50K,0.239282


In [45]:
y = (target == '>50k').astype('int32')
y[:3]

,class
0,0
1,0
2,0


In [46]:
# fnlwgt is a survey weighting variable.
# We omit it from this introductory, unweighted experiment.
x = x.drop(columns=["fnlwgt"])

In [49]:
numeric_columns = x.select_dtypes(include = 'number').columns.tolist()
categorical_columns = x.select_dtypes(exclude = 'number').columns.tolist()
print(numeric_columns)
print(categorical_columns)

['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']


In [51]:
for col in categorical_columns:
  print(f"{col} : {x[col].unique()}")

workclass : ['Private', 'Local-gov', NaN, 'Self-emp-not-inc', 'Federal-gov', 'State-gov', 'Self-emp-inc', 'Without-pay', 'Never-worked']
Categories (8, object): ['Federal-gov', 'Local-gov', 'Never-worked', 'Private', 'Self-emp-inc',
                         'Self-emp-not-inc', 'State-gov', 'Without-pay']
education : ['11th', 'HS-grad', 'Assoc-acdm', 'Some-college', '10th', ..., 'Assoc-voc', '9th', '12th', '1st-4th', 'Preschool']
Length: 16
Categories (16, object): ['10th', '11th', '12th', '1st-4th', ..., 'Masters', 'Preschool',
                          'Prof-school', 'Some-college']
marital-status : ['Never-married', 'Married-civ-spouse', 'Widowed', 'Divorced', 'Separated', 'Married-spouse-absent', 'Married-AF-spouse']
Categories (7, object): ['Divorced', 'Married-AF-spouse', 'Married-civ-spouse',
                         'Married-spouse-absent', 'Never-married', 'Separated', 'Widowed']
occupation : ['Machine-op-inspct', 'Farming-fishing', 'Protective-serv', NaN, 'Other-service', ...,